# Scalable semantic document retrieval with BEIR HotpotQA

This notebook measures how Spark NLP semantic retrieval changes as a Databricks cluster
gains workers. It uses the same real BEIR HotpotQA data as the Colab walkthrough, but
stores data and embeddings in a shared Unity Catalog Volume so every worker can
participate.

The workflow is:

1. Prepare the real HotpotQA corpus in a Unity Catalog Volume once.
2. Select qrel-backed questions and a reproducible corpus sample.
3. Embed passages in parallel with E5 and store embeddings in Delta.
4. Broadcast the small query batch and score query-passage pairs with
   `PairwiseVectorSimilarity`.
5. Rank the top passages, measure Recall@K, and save a benchmark result.

## How to demonstrate scalability

Run this notebook on otherwise identical clusters with **1, 2, 4, and 8 workers**.
Keep the corpus size, query count, top-K, model, and `partition_count` unchanged.
Compare the saved median latency and embedding throughput. Start at 100K passages and
10 queries (1 million exact comparisons), then use the full corpus with 10 queries
(roughly 52 million comparisons).

> This notebook is an **exact-retrieval** benchmark: every selected query is compared
> with every selected passage. It establishes quality and scale-out baselines. For large
> query batches, use BM25, LSH, or ANN candidate generation before semantic reranking.


## 1. Install the Spark NLP artifacts




In [0]:
 %pip install spark-nlp

Processing /Volumes/main/default/libsahmed/spark_nlp-6.4.2-py2.py3-none-any.whl
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


## 2. Import the retrieval libraries




In [0]:
import json
import shutil
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)
from pyspark.sql.window import Window
from sparknlp.annotator import E5Embeddings, PairwiseVectorSimilarity
from sparknlp.base import DocumentAssembler

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


Spark version: 3.5.2
Default parallelism: 16


## 3. Configure the shared Unity Catalog Volume

Update `data_volume` in the next cell to point to a Unity Catalog Volume that is
writable by you. This Volume holds the downloaded HotpotQA files, Delta embeddings,
and benchmark results. It can be the same Volume containing the JAR and wheel.
Do not use restricted `dbfs:/FileStore` storage with this access mode.

You need `READ VOLUME` and `WRITE VOLUME` permissions on this Volume.


In [0]:
# Update this path to your Unity Catalog Volume.
# This Volume holds raw HotpotQA data, Delta embeddings, and benchmark results.
# It can be the same Volume where you uploaded the JAR and wheel.
data_volume = "/Volumes/main/default"

RAW_ROOT = f"{data_volume}/beir/hotpotqa/raw"
DELTA_ROOT = f"{data_volume}/beir/hotpotqa/delta"
RESULTS_PATH = f"{DELTA_ROOT}/benchmark-results"
LOCAL_ARCHIVE = Path("/local_disk0/tmp/hotpotqa.zip")
LOCAL_EXTRACTION = Path("/local_disk0/tmp/beir")
BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/hotpotqa.zip"

print("Shared benchmark data Volume:", data_volume)


Shared benchmark data Volume: /Volumes/main/default/libsahmed


## 4. Download HotpotQA to the shared Unity Catalog Volume once

The first run downloads the official BEIR archive to the driver and copies the three
required files to the configured Unity Catalog Volume. The Volume is shared by the
cluster, so later benchmark runs—on this or another cluster—reuse the same data.
Download time is deliberately excluded from all benchmark measurements.

You need `READ VOLUME` and `WRITE VOLUME` permissions for `data_volume`. If this cell
reports an access error, ask a workspace administrator to grant those permissions or use
a writable Volume. Set `prepare_raw_data=True` only for the first run; afterwards, set it
to `False` to reuse the shared copy.


In [0]:
prepare_raw_data = True

if prepare_raw_data:
    LOCAL_EXTRACTION.mkdir(parents=True, exist_ok=True)
    urlretrieve(BEIR_URL, LOCAL_ARCHIVE)
    with ZipFile(LOCAL_ARCHIVE) as archive:
        archive.extractall(LOCAL_EXTRACTION)
    raw_root_path = Path(RAW_ROOT)
    raw_root_path.mkdir(parents=True, exist_ok=True)
    for relative_path in ("corpus.jsonl", "queries.jsonl", "qrels/test.tsv"):
        source = LOCAL_EXTRACTION / "hotpotqa" / relative_path
        destination = raw_root_path / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

print("HotpotQA is ready in:", RAW_ROOT)


HotpotQA is ready in: /Volumes/main/default/libsahmed/beir/hotpotqa/raw


## 5. Load the searchable passages, questions, and answer key

BEIR supplies `corpus.jsonl` (passages), `queries.jsonl` (questions), and
`qrels/test.tsv` (relevance judgments). We rename their columns to make the retrieval
pipeline easier to read:

- `chunk_id` and `chunk_text`: the searchable passage.
- `query_id` and `query_text`: the search request.
- Positive qrels: passages known to be relevant to a query.


In [0]:
corpus = (
    spark.read.json(f"{RAW_ROOT}/corpus.jsonl")
    .select(
        F.col("_id").alias("chunk_id"),
        F.coalesce(F.col("title"), F.lit("")).alias("title"),
        F.col("text").alias("chunk_text"),
    )
)
queries = (
    spark.read.json(f"{RAW_ROOT}/queries.jsonl")
    .select(F.col("_id").alias("query_id"), F.col("text").alias("query_text"))
)
positive_qrels = (
    spark.read.option("header", True).option("sep", "\t")
    .csv(f"{RAW_ROOT}/qrels/test.tsv")
    .select(
        F.col("query-id").alias("query_id"),
        F.col("corpus-id").alias("chunk_id"),
        F.col("score").cast("int").alias("relevance"),
    )
    .where(F.col("relevance") > 0)
)

print(f"Available passages: {corpus.count():,}")
print(f"Available queries: {queries.count():,}")
print(f"Positive relevance labels: {positive_qrels.count():,}")


Available passages: 5,233,329
Available queries: 97,852
Positive relevance labels: 14,810


## 6. Choose parameters for this benchmark run

Edit the plain values in the next cell . For fair 1-, 2-,
4-, and 8-worker comparisons, keep every value except `run_label` fixed. Keep
`partition_count=128` fixed as well: it gives every cluster the same number of corpus
tasks, while larger clusters can run more of them concurrently.

Use `rebuild_embeddings=True` when measuring distributed embedding throughput. Set it to
`False` for retrieval-only comparisons that reuse the saved Delta embeddings.


In [0]:
corpus_size = "100000"  # Choose "10000", "100000", or "full".
query_count = 10
top_k = 10
partition_count = 128
run_label = "workers-4"
rebuild_embeddings = True

if query_count < 1 or top_k < 1 or partition_count < 1:
    raise ValueError("query_count, top_k, and partition_count must be positive.")

print(
    json.dumps(
        {
            "run_label": run_label,
            "corpus_size": corpus_size,
            "query_count": query_count,
            "top_k": top_k,
            "partition_count": partition_count,
            "rebuild_embeddings": rebuild_embeddings,
        },
        indent=2,
    )
)


{
  "run_label": "workers-4",
  "corpus_size": "100000",
  "query_count": 10,
  "top_k": 10,
  "partition_count": 128,
  "rebuild_embeddings": true
}


## 7. Choose queries with known relevant passages

We select a deterministic batch of questions with positive qrels. Keeping the same query
batch for every worker-count run makes Recall@K and latency directly comparable.


In [0]:
selected_query_ids = (
    positive_qrels.select("query_id").distinct().orderBy("query_id").limit(query_count)
)
selected_queries = (
    queries.join(selected_query_ids, "query_id")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
selected_qrels = (
    positive_qrels.join(selected_query_ids, "query_id")
    .select("query_id", "chunk_id")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

actual_query_count = selected_queries.count()
print(f"Selected queries: {actual_query_count}")
print(f"Positive qrels retained: {selected_qrels.count()}")
display(selected_queries.orderBy("query_id"))


Selected queries: 10
Positive qrels retained: 20


query_id,query_text
5a70eee85542994082a3e3f0,"Which group of people in West Africa with significant populations in Ghana, Ivory Coast, Liberia and Sierra Leone use the Ida?"
5a70f0a75542994082a3e403,"Which dialect spoken in the province of Scania calls Spettekaka, ""spiddekaga""?"
5a70f0e75542994082a3e408,"Which of these universities, Northwestern University or Johns Hopkins University, have a campus outside of the United States territories?"
5a70f11a5542994082a3e40b,"Which dog breed, the Schapendoes or the Bull Terrier, has its origins in a greater number of other dog breeds?"
5a70f1685542994082a3e40f,Which piece did Ludwig van Beethoven publish in 1801 that was dedicated to Count Moritz von Fries?
5a70f39c5542994082a3e429,What is the name of the independent candidate in Maine's 2010 gubernatorial race who finished ahead of Libby Mitchell?
5a70f4695542994082a3e435,"Which canal was built between 1836 and 1847, Washington City Canal or Whitewater Canal?"
5a70f4c45542994082a3e437,"Which Italian-American composer and librettist wrote the English language opera, Maria Golovin?"
5a70f6425542994082a3e44c,"Renamed in 2014, what was the vehicle offered as a prize to contestants on the first season of The Amazing Race Canada?"
5a70f9335542994082a3e46a,"Which magazine is published more in a year, Essence or Alt for Damerne?"


## 8. Build the corpus for this benchmark run

`full` uses every HotpotQA passage. The 10K and 100K modes always retain all passages
judged relevant to the selected queries, then add unrelated passages based on a stable
hash of `chunk_id`. This preserves the answer key while ensuring every rerun receives
the same corpus.


In [0]:
relevant_chunk_ids = selected_qrels.select("chunk_id").distinct()
target_passages = None

if corpus_size == "full":
    benchmark_corpus = corpus.repartition(partition_count)
else:
    target_passages = int(corpus_size)
    relevant_chunks = corpus.join(relevant_chunk_ids, "chunk_id")
    relevant_count = relevant_chunks.count()
    if relevant_count >= target_passages:
        raise ValueError(
            f"{relevant_count} relevant passages exceed requested corpus size {target_passages}."
        )
    distractors = (
        corpus.join(relevant_chunk_ids, "chunk_id", "left_anti")
        .where(F.pmod(F.xxhash64("chunk_id"), F.lit(10_000)) < 500)
        .limit(target_passages - relevant_count)
    )
    benchmark_corpus = relevant_chunks.unionByName(distractors).repartition(
        partition_count
    )

benchmark_corpus = benchmark_corpus.persist(StorageLevel.MEMORY_AND_DISK)
corpus_count = benchmark_corpus.count()
if target_passages and corpus_count != target_passages:
    raise RuntimeError(
        f"Expected {target_passages} passages after sampling, found {corpus_count}."
    )

pair_count = corpus_count * actual_query_count
print(f"Benchmark passages: {corpus_count:,}")
print(f"Queries: {actual_query_count:,}")
print(f"Exact query-passage comparisons per retrieval run: {pair_count:,}")
print(f"Fixed corpus partitions: {benchmark_corpus.rdd.getNumPartitions()}")


Benchmark passages: 100,000
Queries: 10
Exact query-passage comparisons per retrieval run: 1,000,000
Fixed corpus partitions: 128


## 9. Create the E5 embedding pipeline

E5 converts text into numerical embeddings that represent meaning. The `passage: ` and
`query: ` prefixes tell E5 whether the input is searchable content or a search request.
The same fitted pipeline is used for both, so vectors share the same representation.


In [0]:
document_assembler = (
    DocumentAssembler().setInputCol("text").setOutputCol("document")
)
e5_embeddings = (
    E5Embeddings.pretrained("e5_small_v2", "en")
    .setInputCols(["document"])
    .setOutputCol("embeddings")
)
embedding_pipeline = Pipeline(stages=[document_assembler, e5_embeddings])
embedding_model = embedding_pipeline.fit(
    benchmark_corpus.select(
        F.concat(F.lit("passage: "), F.col("chunk_text")).alias("text")
    )
)

print("Embedding model: e5_small_v2")


e5_small_v2 download started this may take some time.
Approximate size to download 76.2 MB
[OK!]
Embedding model: e5_small_v2


## 10. Embed passages and persist the distributed index as Delta

Set `rebuild_embeddings=True` to measure indexing speed. Spark partitions the corpus and
sends embedding work to the cluster executors. The completed vectors are written as a
Delta table, allowing retrieval-only worker-count runs to reuse the exact same vectors.


In [0]:
embedding_path = (
    f"{DELTA_ROOT}/embeddings-{corpus_size}-{query_count}-queries-{partition_count}-partitions"
)
embedding_seconds = None

if rebuild_embeddings:
    embedding_started = time.perf_counter()
    corpus_embeddings_to_save = (
        embedding_model.transform(
            benchmark_corpus.withColumn(
                "text", F.concat(F.lit("passage: "), F.col("chunk_text"))
            )
        )
        .select(
            "chunk_id",
            "title",
            "chunk_text",
            F.col("embeddings").alias("chunk_embeddings"),
        )
    )
    corpus_embeddings_to_save.write.format("delta").mode("overwrite").save(
        embedding_path
    )
    embedding_seconds = time.perf_counter() - embedding_started
    print(f"Corpus embedding time: {embedding_seconds:.3f} seconds")
    print(
        f"Corpus embedding throughput: {corpus_count / embedding_seconds:.2f} passages/second"
    )

corpus_embeddings = (
    spark.read.format("delta").load(embedding_path)
    .repartition(partition_count)
    .persist(StorageLevel.MEMORY_AND_DISK)
)
corpus_embeddings.count()
print(f"Embedding index partitions: {corpus_embeddings.rdd.getNumPartitions()}")


Corpus embedding time: 1498.452 seconds
Corpus embedding throughput: 66.74 passages/second
Embedding index partitions: 128


## 11. Embed the query batch

There are only a few queries, so this phase is normally small. We persist them because
each repeated retrieval benchmark run uses the same query vectors.


In [0]:
query_embedding_started = time.perf_counter()
query_embeddings = (
    embedding_model.transform(
        selected_queries.withColumn(
            "text", F.concat(F.lit("query: "), F.col("query_text"))
        )
    )
    .select(
        "query_id",
        "query_text",
        F.col("embeddings").alias("query_embeddings"),
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)
query_embeddings.count()
query_embedding_seconds = time.perf_counter() - query_embedding_started

print(f"Query embedding time: {query_embedding_seconds:.3f} seconds")
display(query_embeddings.select("query_id", "query_text").orderBy("query_id"))


Query embedding time: 1.602 seconds


query_id,query_text
5a70eee85542994082a3e3f0,"Which group of people in West Africa with significant populations in Ghana, Ivory Coast, Liberia and Sierra Leone use the Ida?"
5a70f0a75542994082a3e403,"Which dialect spoken in the province of Scania calls Spettekaka, ""spiddekaga""?"
5a70f0e75542994082a3e408,"Which of these universities, Northwestern University or Johns Hopkins University, have a campus outside of the United States territories?"
5a70f11a5542994082a3e40b,"Which dog breed, the Schapendoes or the Bull Terrier, has its origins in a greater number of other dog breeds?"
5a70f1685542994082a3e40f,Which piece did Ludwig van Beethoven publish in 1801 that was dedicated to Count Moritz von Fries?
5a70f39c5542994082a3e429,What is the name of the independent candidate in Maine's 2010 gubernatorial race who finished ahead of Libby Mitchell?
5a70f4695542994082a3e435,"Which canal was built between 1836 and 1847, Washington City Canal or Whitewater Canal?"
5a70f4c45542994082a3e437,"Which Italian-American composer and librettist wrote the English language opera, Maria Golovin?"
5a70f6425542994082a3e44c,"Renamed in 2014, what was the vehicle offered as a prize to contestants on the first season of The Amazing Race Canada?"
5a70f9335542994082a3e46a,"Which magazine is published more in a year, Essence or Alt for Damerne?"


## 12. Confirm how Spark will distribute the retrieval work

The corpus has a fixed number of partitions. Spark schedules those partitions as tasks
across available workers; a larger cluster can execute more tasks at the same time. The
query batch is broadcast because it is small, so Spark avoids moving the large embedding
index across the network.

Run the next cell and inspect the Spark UI **Stages** tab while the benchmark runs. The
`BroadcastNestedLoopJoin` in the physical plan is expected: this is the exact
query-passage comparison strategy.


In [0]:
partition_sizes = (
    corpus_embeddings.withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)
print(f"Spark default parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Corpus embedding partitions: {corpus_embeddings.rdd.getNumPartitions()}")
display(partition_sizes)


Spark default parallelism: 16
Corpus embedding partitions: 128


partition_id,count
0,783
1,783
2,783
3,783
4,783
5,782
6,782
7,782
8,782
9,782


## 13. Score and rank every query-passage pair

`PairwiseVectorSimilarity` needs a query embedding and a passage embedding on the same
row. `crossJoin` creates one row for every combination; with 10 queries and 100K
passages this creates one million exact comparisons. Cosine similarity assigns a higher
score to semantically closer pairs. A window then retains the top-K passages **per
query**, rather than one global top-K list.


In [0]:
similarity = (
    PairwiseVectorSimilarity()
    .setInputCols(["query_embeddings", "chunk_embeddings"])
    .setOutputCol("similarity")
    .setSimilarityMethod("cosine")
)
ranking_window = Window.partitionBy("query_id").orderBy(
    F.desc("score"), F.asc("chunk_id")
)


def retrieve_top_k():
    scored_pairs = (
        similarity.transform(F.broadcast(query_embeddings).crossJoin(corpus_embeddings))
        .select(
            "query_id",
            "query_text",
            "chunk_id",
            "title",
            "chunk_text",
            F.explode("similarity").alias("similarity_annotation"),
        )
        .select(
            "query_id",
            "query_text",
            "chunk_id",
            "title",
            "chunk_text",
            F.col("similarity_annotation.result").cast("double").alias("score"),
        )
    )
    return (
        scored_pairs.withColumn("rank", F.row_number().over(ranking_window))
        .where(F.col("rank") <= top_k)
        .orderBy("query_id", "rank")
    )


print(f"Exact comparisons per run: {pair_count:,}")
retrieve_top_k().explain("formatted")


Exact comparisons per run: 1,000,000
== Physical Plan ==
Sort (43)
+- Exchange (42)
   +- * Filter (41)
      +- * RunningWindowFunction (40)
         +- WindowGroupLimit (39)
            +- Sort (38)
               +- Exchange (37)
                  +- WindowGroupLimit (36)
                     +- * Sort (35)
                        +- * Project (34)
                           +- * Generate (33)
                              +- * Project (32)
                                 +- * BroadcastNestedLoopJoin Inner BuildLeft (31)
                                    :- BroadcastExchange (25)
                                    :  +- InMemoryTableScan (1)
                                    :        +- InMemoryRelation (2)
                                    :              +- * Project (24)
                                    :                 +- * SerializeFromObject (23)
                                    :                    +- MapPartitions (22)
                                    :     

## 14. Inspect the retrieved passages

This cell performs one retrieval run and displays the ranked results. The next two
sections evaluate quality and measure repeated warm retrieval latency separately.


In [0]:
ranked_results = retrieve_top_k().persist(StorageLevel.MEMORY_AND_DISK)
ranked_results.count()
display(
    ranked_results.select(
        "query_id", "rank", "score", "title", "chunk_text"
    )
)


query_id,rank,score,title,chunk_text
5a70eee85542994082a3e3f0,1,0.8357010105617351,Ida (sword),"The Ida is a kind of sword used by the Yoruba people of West Africa. It is a long sword with a narrow to wide blade and sheathe. The sword is sharp, and cuts on contact but typically begins to dull if not sharpened regularly. It can be single-edged or double-edged. These blades are typically heavier by the tip of the blade."
5a70eee85542994082a3e3f0,2,0.8305645319328019,Bwa people,"The Bwa or Bwaba (plural), or Bobo-Wule (Bobo-Oule), are an ethnic group indigenous to central Burkina Faso and Mali. Their population is approximately 300,000. They are known for their use of masks, made from leaves or wood, used in performative rituals."
5a70eee85542994082a3e3f0,3,0.823844555528344,Abiriw,"""'Abiriw"" is a town in the Eastern Region of Ghana."
5a70eee85542994082a3e3f0,4,0.8218920412255326,HIV/AIDS in Ivory Coast,"The infection rate of HIV/AIDS in Ivory Coast is estimated at 3,46% in adults ages 15–49. Ivory Coast has a generalized HIV epidemic with the highest prevalence rate in the West African region. The prevalence rate appears to have remained relatively stable for the past decade, with recent declines among pregnant women in urban areas. Civil conflict in the country continues to hinder the collection of new national HIV-related data."
5a70eee85542994082a3e3f0,5,0.8211901986543511,Kwele people,"The Kwele people are a tribal group of eastern Gabon, Republic of the Congo, and Cameroons in West Africa. They fled the coastal area of West Africa during the 19th century, after their traditional enemies acquired firearms from the slave traders. This altercation is often called the ""Poupou"" war. The Kwele then settled into lands between the Dja and Ivindo rivers. The Kwele are noted for their ceremonial masks which are collected as art objects."
5a70eee85542994082a3e3f0,6,0.817446511736989,Grands-Ponts,"Grands-Ponts Region (also originally known as Leboutou Region) is one of the 31 regions of Ivory Coast. Since its establishment in 2011, it has been one of three regions in Lagunes District. The seat of the region is Dabou and the region's population in the 2014 census was 356,495."
5a70eee85542994082a3e3f0,7,0.8151509337637883,Politics of Sierra Leone,"Sierra Leone is a country located in West Africa, known officially as the Republic of Sierra Leone."
5a70eee85542994082a3e3f0,8,0.814935732387725,ID Ghana,"Initiative Development Ghana (or ID Ghana) is a Ghanaian MicroFinance Institution (MFI) based in the Dansoman neighborhood of Accra, Ghana. Founded in 1998, ID Ghana offers loans, voluntary savings, and a host of business support services and financial literacy training to people living in the Greater Accra region. ID Ghana is registered as a Financial NGO (FNGO) in Ghana."
5a70eee85542994082a3e3f0,9,0.8136762629757193,West Africa cricket team,"The West African cricket team was a team representing the countries of Gambia, Ghana, Nigeria and Sierra Leone in international cricket matches whilst they were an associate member of the International Cricket Council between 1976 and 2003. They played in the ICC Trophy on three occasions, in 1982, 1994 and 1997, withdrawing shortly before the start of the 2001 tournament. The team was broken up into its constituent parts in 2003, with Nigeria becoming an associate member of the ICC, the other three affiliates."
5a70eee85542994082a3e3f0,10,0.8120638310126919,Idalia National Park,"Idalia is a national park in South West Queensland, Australia, 893 km west of Brisbane. Idalia National Park is located near the town of Blackall in the Queensland outback. The park protects 144,000 hectares of mulga lands with conservation value. Idalia National Park was opened in 1990 by Prince Phillip."


## 15. Evaluate quality with Recall@K

BEIR's qrels identify passages judged relevant to every selected query. Recall@K is the
fraction of these known relevant passages that appear in the top-K results. Latency
without quality can be misleading, so compare Recall@K alongside every worker-count run.


In [0]:
relevant_per_query = selected_qrels.groupBy("query_id").agg(
    F.countDistinct("chunk_id").alias("relevant_passages")
)
retrieved_relevant = (
    ranked_results.join(selected_qrels, ["query_id", "chunk_id"])
    .groupBy("query_id")
    .agg(F.countDistinct("chunk_id").alias("retrieved_relevant_passages"))
)
recall_by_query = (
    relevant_per_query.join(retrieved_relevant, "query_id", "left")
    .fillna(0, ["retrieved_relevant_passages"])
    .withColumn(
        "recall_at_k",
        F.col("retrieved_relevant_passages") / F.col("relevant_passages"),
    )
)
recall_at_k = recall_by_query.agg(F.avg("recall_at_k").alias("recall")).first()[
    "recall"
]

print(f"Recall@{top_k}: {recall_at_k:.3f}")
display(recall_by_query.orderBy("query_id"))


Recall@10: 0.950


query_id,relevant_passages,retrieved_relevant_passages,recall_at_k
5a70eee85542994082a3e3f0,2,1,0.5
5a70f0a75542994082a3e403,2,2,1.0
5a70f0e75542994082a3e408,2,2,1.0
5a70f11a5542994082a3e40b,2,2,1.0
5a70f1685542994082a3e40f,2,2,1.0
5a70f39c5542994082a3e429,2,2,1.0
5a70f4695542994082a3e435,2,2,1.0
5a70f4c45542994082a3e437,2,2,1.0
5a70f6425542994082a3e44c,2,2,1.0
5a70f9335542994082a3e46a,2,2,1.0


## 16. Benchmark warm exact retrieval

The corpus and query embeddings are cached. One unmeasured warm-up run is discarded;
five measured runs each rebuild the exact cross join, score every pair, and rank top-K.
This excludes model download and embedding/indexing from retrieval latency.


In [0]:
warmup_runs = 1
measured_runs = 5

for _ in range(warmup_runs):
    retrieve_top_k().count()

retrieval_seconds = []
for _ in range(measured_runs):
    retrieval_started = time.perf_counter()
    retrieve_top_k().count()
    retrieval_seconds.append(time.perf_counter() - retrieval_started)

median_seconds = statistics.median(retrieval_seconds)
p95_seconds = max(retrieval_seconds)

print("Distributed exact retrieval benchmark")
print("-" * 40)
print(f"Run label: {run_label}")
print(f"Corpus passages: {corpus_count:,}")
print(f"Queries: {actual_query_count:,}")
print(f"Pairs scored per run: {pair_count:,}")
print(f"Fixed corpus partitions: {partition_count}")
print(f"Measured retrieval runs: {[round(value, 3) for value in retrieval_seconds]}")
print(f"Median retrieval latency: {median_seconds:.3f} seconds")
print(f"p95 retrieval latency: {p95_seconds:.3f} seconds")
print(f"Pair throughput: {pair_count / median_seconds:,.0f} pairs/second")
print(f"Query throughput: {actual_query_count / median_seconds:.2f} queries/second")
print(f"Recall@{top_k}: {recall_at_k:.3f}")


Distributed exact retrieval benchmark
----------------------------------------
Run label: workers-4
Corpus passages: 100,000
Queries: 10
Pairs scored per run: 1,000,000
Fixed corpus partitions: 128
Measured retrieval runs: [13.498, 12.652, 12.918, 13.34, 12.566]
Median retrieval latency: 12.918 seconds
p95 retrieval latency: 13.498 seconds
Pair throughput: 77,409 pairs/second
Query throughput: 0.77 queries/second
Recall@10: 0.950


## 17. Save a benchmark result for cross-cluster comparison

Each run appends one row to a shared Delta table. After running `workers-1`,
`workers-2`, `workers-4`, and `workers-8`, open the table below to compare the median
latency and calculate speedup:

```text
speedup = workers-1 median latency / current median latency
```

Only compare rows with the same corpus size, query count, top-K, model, and partition
count. `embedding_seconds` is null for retrieval-only runs that reused the Delta index.


In [0]:
summary = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "run_label": run_label,
    "corpus_size_setting": corpus_size,
    "corpus_passages": corpus_count,
    "queries": actual_query_count,
    "pairs_scored": pair_count,
    "top_k": top_k,
    "partition_count": partition_count,
    "default_parallelism": spark.sparkContext.defaultParallelism,
    "spark_version": spark.version,
    "embedding_rebuilt": rebuild_embeddings,
    "embedding_seconds": embedding_seconds,
    "query_embedding_seconds": query_embedding_seconds,
    "retrieval_seconds": retrieval_seconds,
    "retrieval_median_seconds": median_seconds,
    "retrieval_p95_seconds": p95_seconds,
    "pair_throughput_per_second": pair_count / median_seconds,
    "query_throughput_per_second": actual_query_count / median_seconds,
    "recall_at_k": recall_at_k,
}

summary_schema = StructType(
    [
        StructField("run_timestamp_utc", StringType(), False),
        StructField("run_label", StringType(), False),
        StructField("corpus_size_setting", StringType(), False),
        StructField("corpus_passages", LongType(), False),
        StructField("queries", LongType(), False),
        StructField("pairs_scored", LongType(), False),
        StructField("top_k", LongType(), False),
        StructField("partition_count", LongType(), False),
        StructField("default_parallelism", LongType(), False),
        StructField("spark_version", StringType(), False),
        StructField("embedding_rebuilt", BooleanType(), False),
        StructField("embedding_seconds", DoubleType(), True),
        StructField("query_embedding_seconds", DoubleType(), False),
        StructField("retrieval_seconds", ArrayType(DoubleType(), False), False),
        StructField("retrieval_median_seconds", DoubleType(), False),
        StructField("retrieval_p95_seconds", DoubleType(), False),
        StructField("pair_throughput_per_second", DoubleType(), False),
        StructField("query_throughput_per_second", DoubleType(), False),
        StructField("recall_at_k", DoubleType(), False),
    ]
)

spark.createDataFrame([summary], schema=summary_schema).write.format("delta").mode("append").save(
    RESULTS_PATH
)
display(
    spark.read.format("delta")
    .load(RESULTS_PATH)
    .where(
        (F.col("corpus_size_setting") == corpus_size)
        & (F.col("queries") == actual_query_count)
        & (F.col("top_k") == top_k)
        & (F.col("partition_count") == partition_count)
    )
    .orderBy(F.desc("run_label"))
)


run_timestamp_utc,run_label,corpus_size_setting,corpus_passages,queries,pairs_scored,top_k,partition_count,default_parallelism,spark_version,embedding_rebuilt,embedding_seconds,query_embedding_seconds,retrieval_seconds,retrieval_median_seconds,retrieval_p95_seconds,pair_throughput_per_second,query_throughput_per_second,recall_at_k
2026-09-01T18:37:05.427357+00:00,workers-8,100000,100000,10,1000000,10,128,32,3.5.2,true,766.6835686530012,1.6089644099993166,"List(7.4083684730012465, 7.596972618001018, 7.931801591001204, 7.309874364998905, 6.781602031000148)",7.4083684730012465,7.931801591001204,134982.48685177564,1.3498248685177565,0.95
2026-09-02T05:20:14.078248+00:00,workers-4,100000,100000,10,1000000,10,128,16,3.5.2,true,1498.451540258,1.6022845490006148,"List(13.497693342003913, 12.651708433004387, 12.918452607998915, 13.340244944003643, 12.566032408001774)",12.918452607998915,13.497693342003913,77408.65182110238,0.7740865182110238,0.95
2026-09-02T04:06:20.928949+00:00,workers-2,100000,100000,10,1000000,10,128,8,3.5.2,true,2802.0400406109984,1.322334119999141,"List(22.420158685999922, 22.246764726995025, 21.437068298000668, 22.057966535998276, 21.358414337002614)",22.057966535998276,22.420158685999922,45335.09461844612,0.4533509461844612,0.9
2026-09-01T18:00:55.657916+00:00,workers-1,100000,100000,10,1000000,10,128,4,3.5.2,true,5808.216586648,1.4520239249995939,"List(41.587336869000865, 41.553626246999556, 41.76752383600069, 41.83274922799865, 42.10881661699932)",41.76752383600069,42.10881661699932,23942.046550963354,0.23942046550963353,0.9


## 18. Scaling checklist

Repeat this notebook with the following settings:

| Run label | Workers | `corpus_size` | `query_count` | `partition_count` | `rebuild_embeddings` |
|---|---:|---:|---:|---:|---|
| `workers-1` | 1 | 100000 | 10 | 128 | `true` |
| `workers-2` | 2 | 100000 | 10 | 128 | `true` |
| `workers-4` | 4 | 100000 | 10 | 128 | `true` |
| `workers-8` | 8 | 100000 | 10 | 128 | `true` |

Then repeat the four runs with `rebuild_embeddings=false` to isolate retrieval latency.
After confirming the 100K runs complete, switch all four to `corpus_size=full`. Watch
the Spark UI during sections 9 and 15: more workers should allow more corpus partitions
to execute concurrently.

Do not compare a worker-count result with different instance types, corpus/query sizes,
partition counts, or embedding settings. For larger query batches, switch to a
two-stage BM25/ANN + `PairwiseVectorSimilarity` reranking design instead of exact
all-pairs retrieval.
